# TSA_ch12_wavelet_coherence

**Published in:** Time Series Analysis  
**Author:** Daniel Traian Pele  
**Description:** Wavelet coherence between two economic series

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt

COLORS = {'blue':'#1A3A6E','red':'#DC3545','green':'#2E7D32','amber':'#B5853F','orange':'#E6802E','purple':'#8E44AD','gray':'#666666'}

In [ ]:
rng = np.random.default_rng(42)
N = 512
t = np.linspace(0, 50, N)

# X: low-frequency oscillation in first half, higher in second
f1 = 0.2; f2 = 0.5
X = np.where(t < 25, np.sin(2*np.pi*f1*t), np.sin(2*np.pi*f2*t)) + 0.3*rng.normal(size=N)
Y = np.where(t < 25, np.sin(2*np.pi*f1*(t-1)), np.sin(2*np.pi*f2*(t-2))) + 0.3*rng.normal(size=N)

wavelet = 'cmor1.5-1.0'
scales = np.geomspace(4, 256, 60)
dt = t[1] - t[0]

cX, _ = pywt.cwt(X, scales, wavelet, sampling_period=dt)
cY, freqs = pywt.cwt(Y, scales, wavelet, sampling_period=dt)

# Smooth cross and auto spectra (Gaussian)
def smooth_2d(W, sigma=2):
    from scipy.ndimage import gaussian_filter
    return gaussian_filter(np.abs(W)**2, sigma=sigma)

Sxx = smooth_2d(cX)
Syy = smooth_2d(cY)
Sxy = np.abs(gaussian_filter := __import__('scipy.ndimage', fromlist=['gaussian_filter']).gaussian_filter)
# Simple smoothed coherence
from scipy.ndimage import gaussian_filter
cross = gaussian_filter(np.real(cX * np.conj(cY)), sigma=2) + \
        1j * gaussian_filter(np.imag(cX * np.conj(cY)), sigma=2)
WCO = (np.abs(cross)**2 / (smooth_2d(cX) * smooth_2d(cY) + 1e-12))
WCO = np.clip(WCO, 0, 1)

fig, axes = plt.subplots(3, 1, figsize=(11, 9), gridspec_kw={'height_ratios': [1, 1, 2.5]})
axes[0].plot(t, X, color=COLORS['blue'], lw=1, label='X')
axes[0].set_title('Series X'); axes[0].set_ylabel('X')
axes[1].plot(t, Y, color=COLORS['red'], lw=1, label='Y')
axes[1].set_title('Series Y'); axes[1].set_ylabel('Y')
im = axes[2].contourf(t, freqs, WCO, levels=20, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[2], label='Wavelet Coherence')
axes[2].set_title('Wavelet Coherence |W_XY|² / (S_XX · S_YY)')
axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_xlabel('Time')
axes[2].set_yscale('log')

for ax in axes:
    ax.set_facecolor('none')
    ax.spines[['top','right']].set_visible(False)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.savefig('wavelet_coherence.pdf', bbox_inches='tight')
plt.show()